# Assignment: The Signal-to-Leverage Pipeline
**Signal Edge, Decay, Utility Maximization, and Leverage**

### Workflow
| Step | Task | Learning Objective |
|------|------|--------------------|
| 1 | Generate alpha signal → Information Coefficient (IC) | Signal Validation |
| 2 | Decay analysis → signal half-life | Decay Awareness |
| 3 | Mean-Variance utility optimisation → position sizes | Risk Management |
| 4 | Apply leverage → monitor risk of ruin | Capital Preservation |

In [60]:
import numpy as np
import pandas as pd
from scipy import stats, optimize
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

SEED        = 42
rng         = np.random.default_rng(SEED)
N_ASSETS    = 20
N_DAYS      = 504        # ~2 years of daily bars
ANNUAL      = 252
SQRT_ANNUAL = np.sqrt(ANNUAL)

# Optimisation / leverage parameters
RISK_AVERSION = 2.5      # lambda in MV utility
TARGET_VOL    = 0.15     # annualised portfolio vol target
MAX_LEVERAGE  = 3.0

# True signal parameters we will try to recover
TRUE_IC       = 0.06
TRUE_HALFLIFE = 15       # days

print('Parameters loaded.')
print(f'  Universe   : {N_ASSETS} assets x {N_DAYS} days')
print(f'  True IC    : {TRUE_IC}')
print(f'  True half-life: {TRUE_HALFLIFE} days')

Parameters loaded.
  Universe   : 20 assets x 504 days
  True IC    : 0.06
  True half-life: 15 days


---
## Step 1 | Alpha Signal Generation & Information Coefficient

The **Information Coefficient (IC)** measures the Spearman rank correlation between a signal value today and the asset return tomorrow:

$$\text{IC}_t = \text{Spearman}\bigl(f_{i,t},\; r_{i,t+1}\bigr)$$

From the **Fundamental Law of Active Management**: $\text{IR} \approx \text{IC} \times \sqrt{\text{Breadth}}$

- A *statistically significant* IC (p < 0.05) and IR > 0.5 indicate genuine predictive power.
- A 'good' IC in quantitative finance is surprisingly low: 0.02 to 0.06.

In [61]:
# ── Asset returns: market factor + idiosyncratic noise ───────────────────────
mkt_ret = rng.normal(0.0003, 0.008, N_DAYS)
betas   = rng.uniform(0.6, 1.4, N_ASSETS)
idio    = rng.normal(0, 0.012, (N_DAYS, N_ASSETS))
returns = betas * mkt_ret[:, None] + idio          # shape (T, N)

# ── Cross-sectional z-score returns ──────────────────────────────────────────
# Raw returns have std ~0.014; noise will have std=1, so we must standardise
# the 'true future return' component so the IC formula is on the same scale.
ret_cs_mean = returns.mean(axis=1, keepdims=True)   # (T,1)
ret_cs_std  = returns.std(axis=1, keepdims=True)    # (T,1)
ret_z       = (returns - ret_cs_mean) / (ret_cs_std + 1e-8)  # (T,N) z-scored

# ── Signal with known IC ─────────────────────────────────────────────────────
# signal[t] = IC * z_return[t+1] + sqrt(1 - IC^2) * noise
# => Pearson(signal[t], ret_z[t+1]) = IC  by construction
# => Spearman IC ≈ IC for large N (ranks preserve linear correlation order)
noise_sig   = rng.normal(0, 1, (N_DAYS, N_ASSETS))
true_next_z = np.vstack([ret_z[1:], ret_z[-1:]])   # shifted z-scored returns
signal      = TRUE_IC * true_next_z + np.sqrt(1 - TRUE_IC**2) * noise_sig

# ── Rolling daily IC (Spearman) ───────────────────────────────────────────────
IC_daily = np.array([
    stats.spearmanr(signal[t], returns[t + 1]).statistic
    for t in range(N_DAYS - 1)
])

IC_mean             = IC_daily.mean()
IC_std              = IC_daily.std(ddof=1)
IC_IR               = IC_mean / IC_std
IC_tstat, IC_pval   = stats.ttest_1samp(IC_daily, 0)

print('IC mean  : {:.4f}  (true = {})'.format(IC_mean, TRUE_IC))
print('IC std   : {:.4f}'.format(IC_std))
print('IC IR    : {:.3f}'.format(IC_IR))
print('T-stat   : {:.2f}   p-value = {:.4f}'.format(IC_tstat, IC_pval))
if IC_pval < 0.05:
    print('Signal is statistically significant (p < 0.05)')
else:
    print('Signal is NOT significant at 5% level')

IC mean  : 0.0520  (true = 0.06)
IC std   : 0.2269
IC IR    : 0.229
T-stat   : 5.14   p-value = 0.0000
Signal is statistically significant (p < 0.05)


In [62]:
ROLL    = 21
ic_roll = pd.Series(IC_daily).rolling(ROLL).mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=np.arange(len(IC_daily)), y=IC_daily,
    name='Daily IC',
    marker_color=np.where(IC_daily >= 0, 'rgba(0,212,170,0.4)', 'rgba(255,107,107,0.4)'),
    marker_line_width=0
))
fig.add_trace(go.Scatter(
    x=np.arange(len(IC_daily)), y=ic_roll,
    mode='lines', name='{}-day Rolling IC'.format(ROLL),
    line=dict(color='#00d4aa', width=2)
))
fig.add_hline(y=IC_mean, line_dash='dash', line_color='lightsalmon',
              annotation_text='Mean IC = {:.3f}'.format(IC_mean),
              annotation_position='top right')
fig.add_hline(y=0, line_color='white', line_width=0.5, opacity=0.3)
fig.add_annotation(
    x=0.02, y=0.95, xref='paper', yref='paper',
    text='IC IR = {:.2f}<br>p-value = {:.4f}'.format(IC_IR, IC_pval),
    showarrow=False, align='left',
    bgcolor='rgba(0,0,0,0.6)', bordercolor='#555',
    font=dict(size=12, color='white')
)
fig.update_layout(
    title='Daily Information Coefficient -- Signal vs. Next-Day Cross-Sectional Returns',
    xaxis_title='Day', yaxis_title='Spearman IC',
    template='plotly_dark'
)
fig.show()

---
## Step 2 | Signal Decay Analysis & Half-Life

Alpha signals decay as information is incorporated into prices. We compute the mean IC at forward horizons $h = 1 \ldots 30$ days and fit an exponential decay:

$$\text{IC}(h) = \text{IC}_0 \cdot e^{-h/\tau}$$

The **half-life** is $t_{1/2} = \tau \ln 2$. It directly dictates the optimal **rebalancing frequency**: rebalancing too slowly wastes edge; too quickly wastes costs.

In [63]:
MAX_HORIZON   = 30
ic_by_horizon = []
for h in range(1, MAX_HORIZON + 1):
    ic_h = np.array([
        stats.spearmanr(signal[t], returns[t + h]).statistic
        for t in range(N_DAYS - h)
    ]).mean()
    ic_by_horizon.append(ic_h)

horizons      = np.arange(1, MAX_HORIZON + 1)
ic_by_horizon = np.array(ic_by_horizon)

# ── Exponential fit ───────────────────────────────────────────────────────────
def exp_decay(h, ic0, tau):
    return ic0 * np.exp(-h / tau)

popt, _ = optimize.curve_fit(exp_decay, horizons, ic_by_horizon,
                              p0=[ic_by_horizon[0], 10], maxfev=5000)
ic0_fit, tau_fit = popt
halflife_fit = tau_fit * np.log(2)

print('Fitted IC0    : {:.4f}  (true ~= {})'.format(ic0_fit, TRUE_IC))
print('Fitted tau    : {:.2f} days'.format(tau_fit))
print('Half-life     : {:.1f} days  (true = {})'.format(halflife_fit, TRUE_HALFLIFE))
print('Recommended rebalance frequency: every {} days'.format(int(np.ceil(halflife_fit))))

Fitted IC0    : 0.5478  (true ~= 0.06)
Fitted tau    : 0.42 days
Half-life     : 0.3 days  (true = 15)
Recommended rebalance frequency: every 1 days


In [64]:
h_fine   = np.linspace(1, MAX_HORIZON, 200)
ic_curve = exp_decay(h_fine, ic0_fit, tau_fit)
hl_ic    = exp_decay(halflife_fit, ic0_fit, tau_fit)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=horizons, y=ic_by_horizon, mode='markers',
    name='Observed IC(h)', marker=dict(color='#00d4aa', size=7)
))
fig.add_trace(go.Scatter(
    x=h_fine, y=ic_curve, mode='lines',
    name='Exp. decay fit (tau={:.1f}d)'.format(tau_fit),
    line=dict(color='lightsalmon', width=2, dash='dash')
))
fig.add_shape(type='line', x0=halflife_fit, x1=halflife_fit, y0=0, y1=hl_ic,
              line=dict(color='cornflowerblue', dash='dot'))
fig.add_shape(type='line', x0=1, x1=halflife_fit, y0=hl_ic, y1=hl_ic,
              line=dict(color='cornflowerblue', dash='dot'))
fig.add_annotation(
    x=halflife_fit, y=hl_ic,
    text='t_half = {:.1f}d'.format(halflife_fit),
    showarrow=True, arrowhead=2, ax=40, ay=-20,
    font=dict(color='cornflowerblue')
)
fig.update_layout(
    title='Signal Decay: IC vs. Forward Horizon',
    xaxis_title='Forward Horizon (days)', yaxis_title='Mean Spearman IC',
    template='plotly_dark'
)
fig.show()

---
## Step 3 | Position Sizing via Mean-Variance Utility

The **Mean-Variance utility** from the presentation:

$$U(w) = w^\top \hat{\mu} - \frac{\lambda}{2} w^\top \Sigma w$$

Unconstrained optimal: $w^* = \frac{1}{\lambda} \Sigma^{-1} \hat{\mu}$

The **alpha forecast** uses the Fundamental Law: $\hat{\mu}_i \propto \text{IC} \cdot \sigma_i \cdot z_i$ where $z_i$ is the z-scored signal. We solve both unconstrained and long-only constrained versions and compare.

In [65]:
# ── Covariance (annualised) ───────────────────────────────────────────────────
ret_df    = pd.DataFrame(returns, columns=['A{:02d}'.format(i) for i in range(N_ASSETS)])
cov_daily = ret_df.cov().values
cov_ann   = cov_daily * ANNUAL
vol_ann   = np.sqrt(np.diag(cov_ann))

# ── Alpha forecast via FLAM ───────────────────────────────────────────────────
signal_today = signal[-1]
z_signal     = (signal_today - signal_today.mean()) / signal_today.std(ddof=1)
mu_hat       = IC_mean * vol_ann * z_signal    # annualised alpha forecast

# ── 1. Unconstrained MV ───────────────────────────────────────────────────────
cov_inv          = np.linalg.inv(cov_ann)
w_unconstrained  = (1.0 / RISK_AVERSION) * cov_inv @ mu_hat

# ── 2. Constrained: long-only, weights sum to 1 ───────────────────────────────
N = N_ASSETS
def neg_utility(w):
    return -(w @ mu_hat - (RISK_AVERSION / 2) * w @ cov_ann @ w)

result = optimize.minimize(
    neg_utility,
    x0=np.ones(N) / N,
    method='SLSQP',
    bounds=[(0, 1)] * N,
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
)
w_constrained = result.x

def portfolio_stats(w, mu, cov):
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    sr  = ret / vol if vol > 0 else 0.0
    lev = np.abs(w).sum()
    return dict(exp_return=ret, vol=vol, sharpe=sr, leverage=lev)

stats_unc = portfolio_stats(w_unconstrained, mu_hat, cov_ann)
stats_con = portfolio_stats(w_constrained,   mu_hat, cov_ann)

print('-- Unconstrained MV --')
for k, v in stats_unc.items(): print('  {:<14}: {:+.4f}'.format(k, v))
print('\n-- Constrained (long-only, sum=1) --')
for k, v in stats_con.items(): print('  {:<14}: {:+.4f}'.format(k, v))

-- Unconstrained MV --
  exp_return    : +0.0295
  vol           : +0.1086
  sharpe        : +0.2716
  leverage      : +2.1870

-- Constrained (long-only, sum=1) --
  exp_return    : +0.0138
  vol           : +0.1291
  sharpe        : +0.1066
  leverage      : +1.0000


In [66]:
assets = ['A{:02d}'.format(i) for i in range(N_ASSETS)]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=assets, y=w_unconstrained, name='Unconstrained MV',
    marker_color='rgba(100,149,237,0.7)'
))
fig.add_trace(go.Bar(
    x=assets, y=w_constrained, name='Constrained (long-only)',
    marker_color='rgba(0,212,170,0.7)'
))
fig.add_hline(y=0, line_color='white', line_width=0.5, opacity=0.3)
fig.update_layout(
    title='Optimal Portfolio Weights: Unconstrained vs. Long-Only MV',
    xaxis_title='Asset', yaxis_title='Weight',
    template='plotly_dark', barmode='group'
)
fig.show()

In [67]:
lambdas = np.logspace(-1, 2, 80)
frontier_vols, frontier_rets = [], []

for lam in lambdas:
    res = optimize.minimize(
        lambda w, lam=lam: -(w @ mu_hat - (lam / 2) * w @ cov_ann @ w),
        x0=np.ones(N) / N,
        method='SLSQP',
        bounds=[(0, 1)] * N,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
    )
    w = res.x
    frontier_vols.append(np.sqrt(w @ cov_ann @ w))
    frontier_rets.append(w @ mu_hat)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=frontier_vols, y=frontier_rets, mode='lines',
    line=dict(color='lightsalmon', width=2),
    name='Efficient frontier'
))
fig.add_trace(go.Scatter(
    x=[stats_con['vol']], y=[stats_con['exp_return']],
    mode='markers',
    marker=dict(size=12, color='#00d4aa', symbol='star'),
    name='Selected (lambda={})'.format(RISK_AVERSION)
))
fig.update_layout(
    title='Mean-Variance Efficient Frontier (long-only, signal-informed)',
    xaxis_title='Annualised Volatility',
    yaxis_title='Expected Return (alpha forecast)',
    template='plotly_dark'
)
fig.show()

---
## Step 4 | Leverage & Risk of Ruin

Leverage amplifies both returns and drawdowns. We apply **volatility-targeting leverage**:

$$L = \min\!\left(\frac{\sigma_{\text{target}}}{\hat{\sigma}_p},\; L_{\text{max}}\right)$$

and monitor **risk of ruin** -- the probability that the portfolio loses more than a threshold $D$ at any point during a one-year horizon -- via Monte Carlo simulation.

> Key insight from the presentation: **Sharpe ratio is leverage-invariant.** Leverage only scales the strategy to a target volatility level; it cannot create alpha.

In [68]:
port_vol_ann = stats_con['vol']
leverage     = min(TARGET_VOL / port_vol_ann, MAX_LEVERAGE)
w_levered    = w_constrained * leverage
stats_lev    = portfolio_stats(w_levered, mu_hat, cov_ann)

print('Base portfolio vol (ann.) : {:.2%}'.format(port_vol_ann))
print('Vol target                : {:.2%}'.format(TARGET_VOL))
print('Applied leverage          : {:.2f}x'.format(leverage))
print('\nLevered portfolio:')
for k, v in stats_lev.items(): print('  {:<14}: {:+.4f}'.format(k, v))

Base portfolio vol (ann.) : 12.91%
Vol target                : 15.00%
Applied leverage          : 1.16x

Levered portfolio:
  exp_return    : +0.0160
  vol           : +0.1500
  sharpe        : +0.1066
  leverage      : +1.1623


In [69]:
lev_range = np.linspace(0.1, MAX_LEVERAGE, 100)
lev_rets, lev_vols, lev_srs = [], [], []

for L in lev_range:
    w = w_constrained * L
    r = w @ mu_hat
    v = np.sqrt(w @ cov_ann @ w)
    lev_rets.append(r)
    lev_vols.append(v)
    lev_srs.append(r / v if v > 0 else 0)

fig = ps.make_subplots(rows=1, cols=2,
                        subplot_titles=['Return & Vol vs. Leverage',
                                        'Sharpe vs. Leverage (invariant)'])
fig.add_trace(go.Scatter(x=lev_range, y=lev_rets, name='Return',
                          line=dict(color='#00d4aa')), row=1, col=1)
fig.add_trace(go.Scatter(x=lev_range, y=lev_vols, name='Volatility',
                          line=dict(color='#ff6b6b')), row=1, col=1)
fig.add_trace(go.Scatter(x=lev_range, y=lev_srs, name='Sharpe',
                          line=dict(color='lightsalmon'), showlegend=False), row=1, col=2)
for col_idx in [1, 2]:
    fig.add_vline(x=leverage, line_dash='dot', line_color='cornflowerblue',
                  annotation_text='L={:.1f}x'.format(leverage),
                  annotation_position='top right',
                  row=1, col=col_idx)
fig.update_xaxes(title_text='Leverage')
fig.update_layout(template='plotly_dark',
                  title='Leverage Sensitivity -- Note Sharpe is Invariant to Leverage',
                  height=400)
fig.show()

### Monte Carlo Risk of Ruin

$$P(\text{ruin}) = P\!\left(\min_t \frac{V_t}{V_0} < 1 - D\right)$$

We simulate 5,000 equity paths over a 1-year horizon using the levered portfolio parameters and record the fraction that breach each drawdown threshold.

In [70]:
MC_PATHS        = 5_000
MC_HORIZON      = ANNUAL
RUIN_THRESHOLDS = [0.10, 0.20, 0.30, 0.50]

port_mu_daily  = (w_levered @ mu_hat) / ANNUAL
port_vol_daily = np.sqrt(w_levered @ cov_daily @ w_levered)

sim_rng   = np.random.default_rng(SEED)
daily_ret = sim_rng.normal(port_mu_daily, port_vol_daily, (MC_PATHS, MC_HORIZON))
equity    = np.cumprod(1 + daily_ret, axis=1)
min_equity = equity.min(axis=1)

print('-- Risk of Ruin (1-year horizon) --')
for d in RUIN_THRESHOLDS:
    prob = (min_equity < 1 - d).mean()
    print('  P(loss > {:.0%}) = {:.2%}'.format(d, prob))

terminal = equity[:, -1]
mc_ret   = terminal - 1
print('\nMC expected annual return : {:.2%}'.format(mc_ret.mean()))
print('MC annual vol             : {:.2%}'.format(mc_ret.std(ddof=1)))
print('MC Sharpe (approx)        : {:.2f}'.format(mc_ret.mean() / mc_ret.std(ddof=1)))

-- Risk of Ruin (1-year horizon) --
  P(loss > 10%) = 44.44%
  P(loss > 20%) = 11.78%
  P(loss > 30%) = 1.14%
  P(loss > 50%) = 0.00%

MC expected annual return : 1.78%
MC annual vol             : 15.15%
MC Sharpe (approx)        : 0.12


In [71]:
days_mc = np.arange(MC_HORIZON)
bands   = np.percentile(equity, [5, 25, 50, 75, 95], axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.concatenate([days_mc, days_mc[::-1]]),
    y=np.concatenate([bands[4], bands[0][::-1]]),
    fill='toself', fillcolor='rgba(100,149,237,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='5th-95th pct'
))
fig.add_trace(go.Scatter(
    x=np.concatenate([days_mc, days_mc[::-1]]),
    y=np.concatenate([bands[3], bands[1][::-1]]),
    fill='toself', fillcolor='rgba(100,149,237,0.25)',
    line=dict(color='rgba(0,0,0,0)'), name='25th-75th pct'
))
fig.add_trace(go.Scatter(
    x=days_mc, y=bands[2], mode='lines',
    line=dict(color='#00d4aa', width=2), name='Median'
))
for d, col in [(0.20, 'rgba(255,107,107,0.6)'), (0.30, 'rgba(255,107,107,0.9)')]:
    fig.add_hline(y=1-d, line_dash='dash', line_color=col,
                  annotation_text='-{:.0%} ruin'.format(d),
                  annotation_position='top left')
fig.add_hline(y=1, line_color='white', line_width=0.5, opacity=0.3)
fig.update_layout(
    title='Monte Carlo Equity Fan Chart -- {:,} Paths, {:.2f}x Leverage'.format(
        MC_PATHS, leverage),
    xaxis_title='Trading Day',
    yaxis_title='Portfolio Value (normalised to 1.0)',
    template='plotly_dark'
)
fig.show()

In [72]:
RUIN_DD   = 0.20
lev_test  = np.linspace(0.5, MAX_LEVERAGE, 30)
ruin_probs = []

for L in lev_test:
    w_l   = w_constrained * L
    mu_d  = (w_l @ mu_hat) / ANNUAL
    vol_d = np.sqrt(w_l @ cov_daily @ w_l)
    dr    = sim_rng.normal(mu_d, vol_d, (2_000, MC_HORIZON))
    eq    = np.cumprod(1 + dr, axis=1)
    ruin_probs.append((eq.min(axis=1) < 1 - RUIN_DD).mean())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lev_test, y=ruin_probs, mode='lines+markers',
    line=dict(color='#ff6b6b', width=2),
    marker=dict(size=5),
    name='P(loss > {:.0%})'.format(RUIN_DD)
))
fig.add_vline(x=leverage, line_dash='dash', line_color='#00d4aa',
              annotation_text='Chosen L={:.2f}x'.format(leverage),
              annotation_position='top right')
fig.update_layout(
    title='Risk of Ruin (>{:.0%} loss) vs. Leverage -- 1-Year Horizon'.format(RUIN_DD),
    xaxis_title='Leverage',
    yaxis_title='Probability of Ruin',
    yaxis_tickformat='.0%',
    template='plotly_dark'
)
fig.show()

---
## Pipeline Summary

In [73]:
summary = {
    'Signal IC (mean)'        : '{:.4f}'.format(IC_mean),
    'IC Information Ratio'    : '{:.3f}'.format(IC_IR),
    'IC T-test p-value'       : '{:.4f}'.format(IC_pval),
    'Signal half-life (days)' : '{:.1f}'.format(halflife_fit),
    'Rebalance frequency'     : 'every {} days'.format(int(np.ceil(halflife_fit))),
    'MV portfolio vol (ann.)' : '{:.2%}'.format(stats_con['vol']),
    'MV portfolio Sharpe'     : '{:.2f}'.format(stats_con['sharpe']),
    'Applied leverage'        : '{:.2f}x'.format(leverage),
    'Levered vol (ann.)'      : '{:.2%}'.format(stats_lev['vol']),
    'Levered Sharpe'          : '{:.2f}'.format(stats_lev['sharpe']),
    'P(loss > 20%, 1yr)'      : '{:.2%}'.format((min_equity < 0.80).mean()),
    'P(loss > 30%, 1yr)'      : '{:.2%}'.format((min_equity < 0.70).mean()),
}
print('\n'.join('  {:<28}: {}'.format(k, v) for k, v in summary.items()))

  Signal IC (mean)            : 0.0520
  IC Information Ratio        : 0.229
  IC T-test p-value           : 0.0000
  Signal half-life (days)     : 0.3
  Rebalance frequency         : every 1 days
  MV portfolio vol (ann.)     : 12.91%
  MV portfolio Sharpe         : 0.11
  Applied leverage            : 1.16x
  Levered vol (ann.)          : 15.00%
  Levered Sharpe              : 0.11
  P(loss > 20%, 1yr)          : 11.78%
  P(loss > 30%, 1yr)          : 1.14%


In [74]:
# Equal-weight baseline (no optimisation, no leverage)
w_ew     = np.ones(N_ASSETS) / N_ASSETS
stats_ew = portfolio_stats(w_ew, mu_hat, cov_ann)

# Panel 1: Sharpe before vs. after MV optimisation
# (leverage is Sharpe-invariant so it belongs in the return panel, not here)
sharpe_labels = ['Equal-weight\n(no optimisation)', 'MV optimised\n(long-only)']
sharpe_values = [stats_ew['sharpe'], stats_con['sharpe']]

# Panel 2: Annualised return before vs. after leverage
ret_labels = ['MV optimised\n(unlevered)', 'MV optimised\n({:.1f}x levered)'.format(leverage)]
ret_values = [stats_con['exp_return'], stats_lev['exp_return']]

fig = ps.make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Step 3: Optimisation lifts Sharpe',
        'Step 4: Leverage amplifies return (Sharpe unchanged)'
    ]
)
fig.add_trace(go.Bar(
    x=sharpe_labels, y=sharpe_values,
    marker_color=['rgba(100,149,237,0.8)', '#00d4aa'],
    text=['{:.2f}'.format(v) for v in sharpe_values],
    textposition='outside', showlegend=False
), row=1, col=1)
fig.add_trace(go.Bar(
    x=ret_labels, y=ret_values,
    marker_color=['lightsalmon', '#00d4aa'],
    text=['{:.1%}'.format(v) for v in ret_values],
    textposition='outside', showlegend=False
), row=1, col=2)
fig.update_yaxes(title_text='Annualised Sharpe', row=1, col=1)
fig.update_yaxes(title_text='Annualised Return', tickformat='.0%', row=1, col=2)
fig.update_layout(
    title='Signal-to-Leverage Pipeline',
    template='plotly_dark', height=420
)
fig.show()

print('Sharpe: equal-weight -> MV optimised : {:.2f} -> {:.2f}'.format(
      stats_ew['sharpe'], stats_con['sharpe']))
print('Return: unlevered -> levered          : {:.1%} -> {:.1%}'.format(
      stats_con['exp_return'], stats_lev['exp_return']))
print('Sharpe before vs. after leverage     : {:.2f} vs {:.2f}  (invariant)'.format(
      stats_con['sharpe'], stats_lev['sharpe']))

Sharpe: equal-weight -> MV optimised : -0.00 -> 0.11
Return: unlevered -> levered          : 1.4% -> 1.6%
Sharpe before vs. after leverage     : 0.11 vs 0.11  (invariant)


---
## Conclusions

### Step 1 -- Signal Validation
A statistically significant IC (p < 0.05) with IC IR > 0.5 indicates genuine predictive power. The Fundamental Law shows that even a modest IC can produce a strong IR if applied across enough independent bets.

### Step 2 -- Decay Awareness
The half-life directly dictates rebalancing frequency. Over-rebalancing on a slow signal increases costs without improving alpha capture; under-rebalancing on a fast signal lets the edge fully decay before being harvested.

### Step 3 -- Utility Maximization
Unconstrained MV takes large short positions, amplifying both alpha and risk. The long-only constrained solution is less theoretically optimal but far more practical. The efficient frontier shows diminishing marginal return for additional concentration beyond the optimal risk-aversion point.

### Step 4 -- Leverage & Risk of Ruin
Volatility targeting provides a principled, systematic leverage rule. Even modest leverage meaningfully raises the probability of severe drawdowns. The Sharpe ratio is leverage-invariant -- the sole purpose of leverage is to hit a target volatility level, not to manufacture alpha.

> **Pipeline takeaway**: IC (edge) -> decay (timing) -> MV utility (sizing) -> vol-target leverage (scaling) -> ruin monitoring forms a complete, auditable chain from raw signal to live position.